In [ ]:
import yaml
import os
from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt
from generative_uncertainty.compute_uncertainty import get_uncertainty_scores 
import os
# Force JAX to only use the CPU backend
os.environ["JAX_PLATFORMS"] = "cpu"
%matplotlib inline
%load_ext autoreload
%autoreload 2

config = "config.yml"
config_data = {}
if os.path.exists(config):
    with open(config, 'r') as f:
        config_data = yaml.safe_load(f) or {} 
else:
    print(f"Warning: Config file '{config}' not found. Using defaults.")

In [ ]:
trained_models_dir = config_data['deep-ensemble']['trained_models_dir']
real_dataset_path = trained_models_dir.format(seed=0) + "/real_dataset.npy"
real_data = jnp.load(real_dataset_path)
print(f"loaded real data : {real_data.shape}")

In [ ]:
samples_cache_dir = config_data['sampling']['samples_cache_dir']
# ensemble_samples_la = jnp.load(Path(samples_cache_dir) / "la_ensemble_samples.npy")
ensemble_samples_deep = jnp.load(Path(samples_cache_dir) / "deep_ensemble_samples.npy")
ensemble_samples_lora = jnp.load(Path(samples_cache_dir) / "lora_ensemble_samples.npy")
# print(f"loaded ensemble samples: {ensemble_samples_la.shape}")
# base_samples_la = ensemble_samples_la[0]
base_samples_deep = ensemble_samples_deep[0]
base_samples_lora = ensemble_samples_lora[0]

uncertainty_scores_deep = get_uncertainty_scores(ensemble_samples_deep)
# uncertainty_scores_la = get_uncertainty_scores(ensemble_samples_la)
uncertainty_scores_lora = get_uncertainty_scores(ensemble_samples_lora)

In [ ]:
from generative_uncertainty.plots import plot_uncertainty_threshold_analysis
plot_uncertainty_threshold_analysis(uncertainty_scores_la)

In [ ]:
plot_uncertainty_threshold_analysis(uncertainty_scores_deep)

In [ ]:
jnp.corrcoef(uncertainty_scores_la, uncertainty_scores_deep)

In [ ]:
uncertainty_scores = uncertainty_scores_lora
base_samples = base_samples_lora
threshold = 77
percentile_score = jnp.percentile(uncertainty_scores, threshold)
print(percentile_score)
confident_mask = uncertainty_scores <= percentile_score
unconfident_mask = uncertainty_scores > percentile_score
filtered_samples = base_samples[confident_mask]
unconfident_samples = base_samples[unconfident_mask]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.5, color='tab:orange')
axes[0].set_title("Train Dataset")

axes[1].scatter(base_samples[:, 0], base_samples[:, 1], s=2, alpha=0.5, color='tab:blue')
axes[1].set_title("Generated Dataset")

axes[2].scatter(filtered_samples[:, 0], filtered_samples[:, 1], s=2, alpha=0.5, color='tab:blue')
# axes[2].scatter(unconfident_samples[:, 0], unconfident_samples[:, 1], s=2, alpha=0.5, color='tab:red')
axes[2].set_title("Filtered Dataset")


for ax in axes:
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    
plt.tight_layout()
plt.show()

In [ ]:
from generative_uncertainty.scoring import extract_true_gmm_params

# 1. Get ground truth distribution properties
true_means, true_var = extract_true_gmm_params()
std_dev = jnp.sqrt(true_var)

# 2. Define a "Ground Truth Hallucination"
# e.g., anything further than 4 standard deviations from its closest mode's center
dist_threshold = 6.0 * std_dev

# 3. Calculate distance to the closest mode for all generated samples
diffs = base_samples[:, None, :] - true_means[None, :, :]
distances = jnp.linalg.norm(diffs, axis=-1)
min_distances = jnp.min(distances, axis=1)

# Ground truth labels
is_true_hallucination = min_distances > dist_threshold

# Predicted labels from your uncertainty percentile (unconfident_mask)
is_pred_hallucination = unconfident_mask 

# 4. Compute Confusion Matrix Masks
TP_mask = is_true_hallucination & is_pred_hallucination   # Rightfully detected (True Positives)
FP_mask = (~is_true_hallucination) & is_pred_hallucination  # Wrongfully detected (False Positives - The "Halos")
TN_mask = (~is_true_hallucination) & (~is_pred_hallucination) # Rightfully kept (True Negatives)
FN_mask = is_true_hallucination & (~is_pred_hallucination)   # Missed hallucinations (False Negatives)

# 5. Plot it!
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(base_samples[TN_mask, 0], base_samples[TN_mask, 1], s=2, alpha=0.3, color='tab:blue', 
           label=f'Rightfully Kept (TN): {TN_mask.sum()}')
ax.scatter(base_samples[FP_mask, 0], base_samples[FP_mask, 1], s=2, alpha=0.3, color='tab:orange', 
           label=f'Wrongly Shaved Halos (FP): {FP_mask.sum()}')
ax.scatter(base_samples[TP_mask, 0], base_samples[TP_mask, 1], s=8, alpha=0.9, color='tab:green', 
           label=f'Rightfully Caught (TP): {TP_mask.sum()}')
ax.scatter(base_samples[FN_mask, 0], base_samples[FN_mask, 1], s=8, alpha=0.9, color='tab:red', 
           label=f'Missed Hallucinations (FN): {FN_mask.sum()}')

# Draw circles around the modes to visualize the threshold
for mu in true_means:
    circle = plt.Circle((mu[0], mu[1]), dist_threshold, color='black', fill=False, linestyle='--', alpha=0.3)
    ax.add_patch(circle)

ax.set_title(f"Predicted vs True Hallucinations (at {threshold}%)")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize='large')
# ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.5, color='tab:orange')
axes[0].set_title("Train Dataset")

axes[1].scatter(base_samples_la[:, 0], base_samples_la[:, 1], s=2, alpha=0.5, color='tab:blue')
axes[1].set_title("Generated Dataset")

for ax in axes:
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    
plt.tight_layout()
plt.show()

In [ ]:

percentiles = [70, 77, 80]

fig, axes = plt.subplots(1, len(percentiles), figsize=(12, 4))

for i, percentile in enumerate(percentiles):
    percentile_score = jnp.percentile(uncertainty_scores_deep, percentile)
    confident_mask = uncertainty_scores_deep <= percentile_score
    filtered_samples = base_samples_deep[confident_mask]
    axes[i].scatter(filtered_samples[:, 0], filtered_samples[:, 1], s=2, alpha=0.5, color='tab:blue')
    axes[i].set_title(f"Filtered Dataset ({percentile}th Percentile)")

for ax in axes:
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    
plt.tight_layout()
plt.show()

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = ensemble_samples_deep
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 6
ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()

In [ ]:
jnp.where(unconfident_mask)

In [ ]:
jnp.where(FP_mask)

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = ensemble_samples_lora
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 6
target_point = 9101
ax.scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.01, color='tab:orange')
# ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
for model_id in range(ensemble_samples.shape[0]):
    ax.scatter(ensemble_samples[model_id, target_point, 0], ensemble_samples[model_id, target_point, 1], s=100, alpha=0.9, label=f'Model {model_id}', marker='x')
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()
plt.legend()

In [ ]:
from generative_uncertainty.scoring import extract_true_gmm_params, evaluate_exact_gmm, gmm_score_percentiles
true_means, true_var = extract_true_gmm_params()

In [ ]:
df_la_gmm = gmm_score_percentiles(real_data, ensemble_samples_la, uncertainty_scores_la, percentile_step=3)
df_deep_gmm = gmm_score_percentiles(real_data, ensemble_samples_deep, uncertainty_scores_deep, percentile_step=3)
df_lora_gmm = gmm_score_percentiles(real_data, ensemble_samples_lora, uncertainty_scores_lora, percentile_step=3)
real_data_baseline = evaluate_exact_gmm(real_data, true_means, true_var)

In [ ]:
import numpy as np
import pandas as pd
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Generative Uncertainty Filtering Performance Across Percentiles", fontsize=16)
axes[0].plot(df_deep_gmm.columns, df_deep_gmm.loc['avg_log_likelihood'], marker='o', label='Generated Data (DE-Filter)')
axes[1].plot(df_deep_gmm.columns, df_deep_gmm.loc['mode_kl_divergence'], marker='o', label='Generated Data (DE-Filter)')
axes[2].plot(df_deep_gmm.columns, df_deep_gmm.loc['variance_ratio'], marker='o', label='Generated Data (DE-Filter)')
# axes[2].fill_between(np.asarray(df_deep_mmd.columns), pd.to_numeric(df_deep_mmd.loc['ci_lower']), pd.to_numeric(df_deep_mmd.loc['ci_upper']), color='tab:blue', alpha=0.2, label='95% CI (DE-Filter)')

axes[0].plot(df_la_gmm.columns, df_la_gmm.loc['avg_log_likelihood'], marker='x', label='Generated Data (LA-Filter)')
axes[1].plot(df_la_gmm.columns, df_la_gmm.loc['mode_kl_divergence'], marker='x', label='Generated Data (LA-Filter)')
axes[2].plot(df_la_gmm.columns, df_la_gmm.loc['variance_ratio'], marker='x', label='Generated Data (LA-Filter)')

axes[0].plot(df_lora_gmm.columns, df_lora_gmm.loc['avg_log_likelihood'], marker='s', label='Generated Data (LoRA-Filter)')
axes[1].plot(df_lora_gmm.columns, df_lora_gmm.loc['mode_kl_divergence'], marker='s', label='Generated Data (LoRA-Filter)')
axes[2].plot(df_lora_gmm.columns, df_lora_gmm.loc['variance_ratio'], marker='s', label='Generated Data (LoRA-Filter)')

# axes[2].fill_between(np.asarray(df_la_mmd.columns), pd.to_numeric(df_la_mmd.loc['ci_lower']), pd.to_numeric(df_la_mmd.loc['ci_upper']), color='tab:orange', alpha=0.2, label='95% CI (LA-Filter)')
axes[0].axhline(real_data_baseline['avg_log_likelihood'], color='tab:red', linestyle='--', label='Real Data')
axes[1].axhline(real_data_baseline['mode_kl_divergence'], color='tab:red', linestyle='--', label='Real Data')
axes[2].axhline(real_data_baseline['variance_ratio'], color='tab:red', linestyle='--', label='Real Data')
# axes[2].axhline(ref_mmd['mean'], color='tab:orange', linestyle='--', label='Real Data')
# axes[2].fill_between([70, 100], ref_mmd['ci_lower'], ref_mmd['ci_upper'], color='tab:orange', alpha=0.2, label='95% CI (Real Data)')
axes[0].set_xlabel('Percentile')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title(r'Log-Likelihood (Precision) ($\uparrow$)')
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title(r'Mode KL Divergence (Recall) ($\downarrow$)')
axes[2].set_xlabel('Percentile')
axes[2].set_ylabel('Variance Ratio')
axes[2].set_title(r'Variance Ratio ($\uparrow$)')
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
axes[2].grid(True, alpha=0.3)
# axes[1].set_xlim(80, 100)
axes[1].set_ylim(-0.001)
axes[0].legend()
axes[1].legend()
axes[2].legend()

In [ ]:
from generative_uncertainty.scoring import mmd_score_percentiles
from generative_uncertainty.scoring import estimator, single_rbf_mmd

In [ ]:
percentiles = [70, 72, 75, 76, 77, 78, 79, 80, 85, 90, 95, 100]
mmd_df_deep = mmd_score_percentiles(real_data, ensemble_samples_deep, uncertainty_scores_deep, gamma=0.2, percentiles=percentiles, num_iterations=30, subsample_size=10000)
ref_mmd = estimator(lambda X, Y: single_rbf_mmd(X, Y, gamma=0.2), real_data, real_data, num_iterations=30, subsample_size=10000)

In [ ]:
saved_df = pd.DataFrame(mmd_df_deep)
saved_df.to_csv("mmd_deep_percentiles.csv", index=False)

In [ ]:
import pandas as pd
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mmd_df_deep.columns, mmd_df_deep.loc['mean'], marker='o', label='Generated Data (DE-Filter)')
ax.fill_between(mmd_df_deep.columns, pd.to_numeric(mmd_df_deep.loc['ci_lower']), pd.to_numeric(mmd_df_deep.loc['ci_upper']), color='tab:blue', alpha=0.2, label='95% CI (DE-Filter)')
ax.axhline(ref_mmd['mean'], color='tab:orange', linestyle='--', label='Real Data')
ax.fill_between(mmd_df_deep.columns, ref_mmd['ci_lower'], ref_mmd['ci_upper'], color='tab:orange', alpha=0.2, label='95% CI (Real Data)')   
ax.set_xlabel('Percentile')
ax.set_ylabel('MMD Score')
ax.set_title('MMD Score Across Percentiles (DE-Filter)')
ax.grid(True, alpha=0.3)
ax.legend()

In [ ]:
import glob
# os.listdir("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples")
samples = glob.glob("/dtu/blackhole/13/213811/s243425/gaussian_experiment/samples/la*.npy")
for i, sample in enumerate(samples):
    print(f"{i}: {sample}")

In [ ]:
def get_clean_label(filepath):
    name = Path(filepath).stem
    prefix = "la_ensemble_samples_"
    if name.startswith(prefix):
        name = name[len(prefix):]
    if not name or name == "la_ensemble_samples":
        return "LA (Default)"
        
    name = name.replace('prior', 'Prior: ')
    name = name.replace('_approx', ', Approx: ')
    name = name.replace('_curv', ', Curv: ')
    name = name.replace('_subset', ', Subset: ')
    name = name.replace('_m', ', m: ')
    return name

get_clean_label(samples[0])

In [ ]:
runs_selected = [0, 1, 11, 12, 13]
runs = []
for run in runs_selected:
    ensemble_samples = jnp.load(samples[run])
    print(f"Run {run} - Ensemble Samples Shape: {ensemble_samples.shape}")
    base_samples = ensemble_samples[0]
    uncertainty_scores = get_uncertainty_scores(ensemble_samples)
    print(f"Run {run} - Uncertainty Scores Shape: {uncertainty_scores.shape}")
    runs.append({"label": get_clean_label(samples[run]), "uncertainty_scores": uncertainty_scores, "ensemble_samples": ensemble_samples})

In [ ]:
# plot of samples from one of the ensemble members
ensemble_samples = runs[3]['ensemble_samples']
fig, ax = plt.subplots(figsize=(6, 6))
model_id = 5
ax.scatter(ensemble_samples[model_id, :, 0], ensemble_samples[model_id, :, 1], s=2, alpha=0.5, color='tab:blue')
ax.set_title("Generated Dataset")
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from generative_uncertainty.scoring import uncertainty_alignment_aurc

rejection_rates = np.linspace(0.0, 0.95, 96)


alignment = None


fig, ax = plt.subplots(figsize=(9, 5))

for run in runs:
    alignment = uncertainty_alignment_aurc(
        reference_uncertainty=uncertainty_scores_deep,
        approx_uncertainty=run["uncertainty_scores"],
        rejection_rates=rejection_rates,
    )

    ax.plot(
        alignment["candidate"]["rejection_rates"],
        alignment["candidate"]["risks"],
        label=f"{run['label']} (AURC={alignment['candidate']['aurc']:.6f} )",
        linewidth=2,
    )

ax.plot(
    alignment["oracle"]["rejection_rates"],
    alignment["oracle"]["risks"],
    label=f"Deep ranking oracle (AURC={alignment['oracle']['aurc']:.6f})",
    linewidth=2,
    linestyle="--",
)
ax.plot(
    alignment["random"]["rejection_rates"],
    alignment["random"]["risks"],
    label=f"Random ranking (AURC={alignment['random']['aurc']:.6f})",
    linewidth=2,
    color="tab:gray",
    linestyle=":",
)
ax.set_xlabel("Rejection Rate")
ax.set_ylabel("Mean Deep-Uncertainty of Kept Samples")
ax.set_title("LA Uncertainty vs Deep-Ensemble Baseline")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

aurc_gap_to_oracle = alignment["candidate"]["aurc"] - alignment["oracle"]["aurc"]
print(f"Deep-oracle AURC: {alignment['oracle']['aurc']:.6f}")
print(f"LA AURC:        {alignment['candidate']['aurc']:.6f}")
print(f"AURC gap to oracle: {aurc_gap_to_oracle:.6f} (closer to 0 is better)")
print(f"Pearson r:  {alignment['metrics']['pearson_r']:.6f}")
print(f"Spearman r: {alignment['metrics']['spearman_r']:.6f}")
print(f"MAE (norm): {alignment['metrics']['mae_normalized']:.6f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from generative_uncertainty.scoring import uncertainty_alignment_aurc

rejection_rates = np.linspace(0.0, 0.95, 96)


alignment = None


fig, ax = plt.subplots(figsize=(9, 5))

alignment = uncertainty_alignment_aurc(
    reference_uncertainty=uncertainty_scores_deep,
    approx_uncertainty=uncertainty_scores_lora,
    rejection_rates=rejection_rates,
)

ax.plot(
    alignment["candidate"]["rejection_rates"],
    alignment["candidate"]["risks"],
    label=f"(AURC={alignment['candidate']['aurc']:.6f} )",
    linewidth=2,
)

ax.plot(
    alignment["oracle"]["rejection_rates"],
    alignment["oracle"]["risks"],
    label=f"Deep ranking oracle (AURC={alignment['oracle']['aurc']:.6f})",
    linewidth=2,
    linestyle="--",
)
ax.plot(
    alignment["random"]["rejection_rates"],
    alignment["random"]["risks"],
    label=f"Random ranking (AURC={alignment['random']['aurc']:.6f})",
    linewidth=2,
    color="tab:gray",
    linestyle=":",
)
ax.set_xlabel("Rejection Rate")
ax.set_ylabel("Mean Deep-Uncertainty of Kept Samples")
ax.set_title("LA Uncertainty vs Deep-Ensemble Baseline")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

aurc_gap_to_oracle = alignment["candidate"]["aurc"] - alignment["oracle"]["aurc"]
print(f"Deep-oracle AURC: {alignment['oracle']['aurc']:.6f}")
print(f"LA AURC:        {alignment['candidate']['aurc']:.6f}")
print(f"AURC gap to oracle: {aurc_gap_to_oracle:.6f} (closer to 0 is better)")
print(f"Pearson r:  {alignment['metrics']['pearson_r']:.6f}")
print(f"Spearman r: {alignment['metrics']['spearman_r']:.6f}")
print(f"MAE (norm): {alignment['metrics']['mae_normalized']:.6f}")